# EXP048 post-hoc tail diagnostic — where EXP047 helped and EXP048 broke ranking

**No training. No validation-driven checkpoint selection.**

Goal: test the concrete hypothesis that the useful hard-negative signal may live in a
**broader middle of the top tail (especially 0.2–1%)**, while the top 0.1% may be too
special to deserve EXP048's 65× per-position emphasis.

Fine train-negative rank bands are frozen from saved **EXP045 train logits**:

- `0–0.1%`
- `0.1–0.2%`
- `0.2–0.5%`
- `0.5–1%`
- `1–5%`
- `5–15%`
- `15–100%` control

The diagnostic uses a deterministic uniform coordinate sample of **1,000 negatives per band**.
It also samples **1,000 train positives from each EXP045-score quartile** to detect collateral
damage to weak positives.

The exact same sampled coordinates are scored by:

- EXP045 — already stored in corrected train shards, **no inference**;
- EXP047-B — adopted risk-focused baseline;
- EXP048 — ultra-tail candidate.

We then measure:

- score shifts and threshold occupancy by negative band;
- population-weighted negative BCE;
- score shifts of positive quartiles;
- AUC / inversion rate of every negative band against all positives and against **weak Q1 positives**;
- a dedicated natural-weighted `0.2–1%` summary;
- existing validation HARD_FP / LOW_TP fixed cohorts, with **no new validation inference**.

This is a diagnostic. It does not automatically define EXP049.


In [1]:
from pathlib import Path
import hashlib, os, subprocess, sys, json
import pandas as pd
import numpy as np
from IPython.display import display

EXPECTED_PYTHON = Path(r"C:\ML_Progs\Runtimes\Miniforge3\envs\epic-eda\python.exe").resolve()
assert Path(sys.executable).resolve() == EXPECTED_PYTHON, (sys.executable, EXPECTED_PYTHON)

CURRENT = Path.cwd().resolve()
PROJECT_ROOT = next(
    c for c in (CURRENT, *CURRENT.parents)
    if (c / "README.md").is_file() and (c / "src/ml_project").is_dir()
)

AGENTS = next(
    (p for p in (PROJECT_ROOT/"AGENTS.md", PROJECT_ROOT.parent/"AGENTS.md") if p.is_file()),
    None,
)
assert AGENTS is not None, "AGENTS.md not found"
print("AGENTS.md:", AGENTS)
print(AGENTS.read_text(encoding="utf-8"))

TARGET_SCRIPT = PROJECT_ROOT / "scripts/run_exp048_tail_diagnostic.py"
SCRIPT_SOURCE = 'from __future__ import annotations\n\nimport argparse\nimport gc\nimport hashlib\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport sys\nimport time\nimport warnings\n\nimport numpy as np\nimport pandas as pd\n\nKEY_SHIFT = np.uint64(56)\nREGION_MASK = (np.uint64(1) << KEY_SHIFT) - np.uint64(1)\nBIT_SHIFTS = np.arange(32, dtype=np.uint32)\nSCORE_LEVELS = 100_001\n\nNEGATIVE_BAND_EDGES = np.asarray([0.001, 0.002, 0.005, 0.01, 0.05, 0.15], dtype=np.float64)\nNEGATIVE_BAND_NAMES = [\n    "0-0.1%",\n    "0.1-0.2%",\n    "0.2-0.5%",\n    "0.5-1%",\n    "1-5%",\n    "5-15%",\n    "15-100%",\n]\nNEGATIVE_SAMPLE_PER_BAND = 1_000\nPOSITIVE_SAMPLE_PER_QUARTILE = 1_000\nRNG_SEED = np.uint64(480048)\nGPU_DUTY_CYCLE = float(os.environ.get("EPIC_GPU_DUTY_CYCLE", "0.90"))\nif not (0.50 <= GPU_DUTY_CYCLE <= 1.0):\n    raise RuntimeError(f"EPIC_GPU_DUTY_CYCLE must be in [0.50, 1.0], got {GPU_DUTY_CYCLE}")\n\n\ndef project_root() -> Path:\n    here = Path(__file__).resolve()\n    for candidate in (here.parent, *here.parents):\n        if (candidate / "README.md").is_file() and (candidate / "src/ml_project").is_dir():\n            return candidate\n    raise RuntimeError("Project root was not found")\n\n\nROOT = project_root()\nSRC = ROOT / "src"\nif str(SRC) not in sys.path:\n    sys.path.insert(0, str(SRC))\n\nfrom ml_project.epic_baseline import load_baseline_sequences\nfrom ml_project.epic_cnn import (\n    ProfileWindowConfig,\n    ProfileWindowGenerator,\n    create_cnn_presence_intensity,\n    forward_both_strands_multitask,\n)\nfrom ml_project.epic_data import load_split\n\nBASE_RUN = (\n    ROOT / "artifacts" / "experiments"\n    / "CNN_EXP045_WIDTH512_50K_TWO_PHASE_LR_RF1029" / "run_001"\n)\nBASE_CHECKPOINT = BASE_RUN / "model_step_50000.pt"\n\nEXP046_RUN = (\n    ROOT / "artifacts" / "experiments"\n    / "CNN_EXP046B_FROZEN_REGIONAL_CORRECTION_R32" / "run_001"\n)\nCV_SHARDS = EXP046_RUN / "_cv_eval_v3"\n\nEXP047_RUN = (\n    ROOT / "artifacts" / "experiments"\n    / "CNN_EXP047_WIDTH512_PAIRED_RISK_LOSS_CONTINUATION" / "run_001"\n)\nEXP047_CHECKPOINT = EXP047_RUN / "branch_B" / "branch_B_final.pt"\nEXP047_COHORT = EXP047_RUN / "branch_B" / "fixed_cohort_scores.csv"\n\nEXP048_RUN = (\n    ROOT / "artifacts" / "experiments"\n    / "CNN_EXP048_WIDTH512_ULTRA_TAIL_RISK_REFINEMENT" / "run_001"\n)\nEXP048_CHECKPOINT = EXP048_RUN / "branch_B" / "branch_B_final.pt"\nEXP048_COHORT = EXP048_RUN / "branch_B" / "fixed_cohort_scores.csv"\n\nRUN_DIR = (\n    ROOT / "artifacts" / "experiments"\n    / "CNN_EXP048_TAIL_DIAGNOSTIC" / "run_001"\n)\nSAMPLE_DIR = RUN_DIR / "sample"\nSAMPLE_MANIFEST = SAMPLE_DIR / "train_tail_sample.csv"\nSAMPLE_META = SAMPLE_DIR / "train_tail_sample_meta.json"\nSCORES_047 = RUN_DIR / "sample_scores_EXP047B.csv"\nSCORES_048 = RUN_DIR / "sample_scores_EXP048.csv"\n\n\ndef atomic_json(path: Path, payload: dict) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(path.suffix + ".tmp")\n    with open(tmp, "w", encoding="utf-8", newline="\\n") as handle:\n        json.dump(payload, handle, ensure_ascii=False, indent=2, allow_nan=False)\n        handle.write("\\n")\n        handle.flush()\n        os.fsync(handle.fileno())\n    os.replace(tmp, path)\n\n\ndef atomic_csv(path: Path, frame: pd.DataFrame) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(path.suffix + ".tmp")\n    frame.to_csv(tmp, index=False)\n    with open(tmp, "r+b") as handle:\n        os.fsync(handle.fileno())\n    os.replace(tmp, path)\n\n\ndef sha256_file(path: Path) -> str:\n    digest = hashlib.sha256()\n    with open(path, "rb") as handle:\n        for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef sigmoid_numpy(logits: np.ndarray) -> np.ndarray:\n    logits = np.asarray(logits, dtype=np.float64)\n    out = np.empty_like(logits)\n    pos = logits >= 0\n    out[pos] = 1.0 / (1.0 + np.exp(-logits[pos]))\n    ex = np.exp(logits[~pos])\n    out[~pos] = ex / (1.0 + ex)\n    return out\n\n\ndef score_bins(logits: np.ndarray) -> np.ndarray:\n    return np.rint(np.clip(sigmoid_numpy(logits), 0.0, 1.0) * 100_000).astype(np.int32)\n\n\ndef unpack_mask(mask_u32: np.ndarray) -> np.ndarray:\n    values = np.asarray(mask_u32, dtype=np.uint32)\n    return ((values[:, None] >> BIT_SHIFTS[None, :]) & np.uint32(1)).astype(bool)\n\n\ndef iter_shards() -> list[Path]:\n    files: list[Path] = []\n    for fold in sorted(path for path in CV_SHARDS.glob("fold_*") if path.is_dir()):\n        files.extend(sorted((fold / "shards").glob("tiles_*.npz")))\n    if not files:\n        raise FileNotFoundError(f"No corrected train shards under {CV_SHARDS}")\n    return files\n\n\ndef splitmix64(x: np.ndarray) -> np.ndarray:\n    x = np.asarray(x, dtype=np.uint64)\n    x = x + np.uint64(0x9E3779B97F4A7C15)\n    z = x.copy()\n    z = (z ^ (z >> np.uint64(30))) * np.uint64(0xBF58476D1CE4E5B9)\n    z = (z ^ (z >> np.uint64(27))) * np.uint64(0x94D049BB133111EB)\n    return z ^ (z >> np.uint64(31))\n\n\ndef coordinate_hash(region_key: np.ndarray, offset: np.ndarray, salt: int) -> np.ndarray:\n    key = np.asarray(region_key, dtype=np.uint64)\n    off = np.asarray(offset, dtype=np.uint64)\n    mixed = key ^ (off * np.uint64(0xD6E8FEB86659FD93)) ^ np.uint64(salt)\n    return splitmix64(mixed)\n\n\ndef _trim_smallest(frame: pd.DataFrame, k: int) -> pd.DataFrame:\n    if len(frame) <= k:\n        return frame\n    values = frame["_hash"].to_numpy(np.uint64)\n    indices = np.argpartition(values, k - 1)[:k]\n    return frame.iloc[indices].copy()\n\n\ndef _decode_coordinates(keys: np.ndarray, offsets: np.ndarray, all_contigs: np.ndarray):\n    keys = np.asarray(keys, dtype=np.uint64)\n    offsets = np.asarray(offsets, dtype=np.int64)\n    prefix = keys >> KEY_SHIFT\n    contig_codes = (prefix // np.uint64(2)).astype(np.int64)\n    strand_codes = (prefix % np.uint64(2)).astype(np.int64)\n    region_ids = (keys & REGION_MASK).astype(np.int64)\n    contigs = all_contigs[contig_codes]\n    strands = np.where(strand_codes == 0, "+", "-")\n    positions = region_ids * 32 + offsets\n    return contigs, strands, positions\n\n\ndef build_sample() -> None:\n    required = [BASE_CHECKPOINT, EXP047_CHECKPOINT, EXP048_CHECKPOINT, CV_SHARDS]\n    missing = [p for p in required if not p.exists()]\n    if missing:\n        raise FileNotFoundError(missing)\n\n    if SAMPLE_MANIFEST.exists() and SAMPLE_META.exists():\n        meta = json.loads(SAMPLE_META.read_text(encoding="utf-8"))\n        if meta.get("complete"):\n            print(f"Sample already complete: {SAMPLE_MANIFEST}")\n            print(json.dumps(meta, indent=2))\n            return\n\n    RUN_DIR.mkdir(parents=True, exist_ok=True)\n    SAMPLE_DIR.mkdir(parents=True, exist_ok=True)\n\n    split = load_split(ROOT)\n    train_contigs = sorted(set(split.intervals("train")["contig"].astype(str)))\n    validation_contigs = sorted(set(split.intervals("validation")["contig"].astype(str)))\n    all_contigs = np.asarray(train_contigs + validation_contigs, dtype=object)\n\n    files = iter_shards()\n    histogram = np.zeros(SCORE_LEVELS, dtype=np.int64)\n    positive_keys_parts = []\n    positive_offsets_parts = []\n    positive_logits_parts = []\n\n    started = time.perf_counter()\n    for i, path in enumerate(files, start=1):\n        with np.load(path, allow_pickle=False) as data:\n            valid = unpack_mask(data["valid_mask"])\n            positive = unpack_mask(data["positive_mask"])\n            negative = valid & ~positive\n            logits = data["base_logit"]\n            bins = score_bins(logits)\n            histogram += np.bincount(bins[negative], minlength=SCORE_LEVELS)\n\n            pos_rows, pos_offsets = np.nonzero(positive)\n            if len(pos_rows):\n                positive_keys_parts.append(\n                    data["region_key"][pos_rows].astype(np.uint64, copy=False)\n                )\n                positive_offsets_parts.append(pos_offsets.astype(np.uint8, copy=False))\n                positive_logits_parts.append(\n                    logits[pos_rows, pos_offsets].astype(np.float32, copy=False)\n                )\n        if i == 1 or i % 500 == 0 or i == len(files):\n            print(f"histogram + positives: {i:,}/{len(files):,}", flush=True)\n\n    total_negative = int(histogram.sum())\n    if total_negative <= 0:\n        raise AssertionError("No train negatives found")\n\n    cut_counts = np.asarray(\n        [round(total_negative * x) for x in NEGATIVE_BAND_EDGES], dtype=np.int64\n    )\n    population_counts = np.diff(\n        np.r_[np.int64(0), cut_counts, np.int64(total_negative)]\n    )\n    population_fractions = population_counts.astype(np.float64) / total_negative\n\n    higher = np.cumsum(histogram[::-1], dtype=np.int64)[::-1] - histogram\n    seen_by_score = np.zeros(SCORE_LEVELS, dtype=np.int64)\n\n    reservoirs: dict[int, pd.DataFrame] = {\n        i: pd.DataFrame() for i in range(len(NEGATIVE_BAND_NAMES))\n    }\n    realized = np.zeros(len(NEGATIVE_BAND_NAMES), dtype=np.int64)\n\n    for file_index, path in enumerate(files, start=1):\n        with np.load(path, allow_pickle=False) as data:\n            valid = unpack_mask(data["valid_mask"])\n            positive = unpack_mask(data["positive_mask"])\n            negative = valid & ~positive\n            logits = data["base_logit"]\n            bins = score_bins(logits)\n\n            neg_rows, neg_offsets = np.nonzero(negative)\n            negative_scores = bins[neg_rows, neg_offsets]\n            if len(negative_scores) == 0:\n                continue\n\n            order = np.argsort(negative_scores, kind="stable")\n            ordered_scores = negative_scores[order]\n            group_starts = np.r_[\n                np.int64(0),\n                np.flatnonzero(ordered_scores[1:] != ordered_scores[:-1]).astype(np.int64) + 1,\n            ]\n            group_ends = np.r_[group_starts[1:], np.int64(len(ordered_scores))]\n            group_lengths = group_ends - group_starts\n            unique_scores = ordered_scores[group_starts]\n            offsets_in_score_group = np.arange(len(ordered_scores), dtype=np.int64) - np.repeat(\n                group_starts, group_lengths\n            )\n            rank_bases = np.repeat(\n                higher[unique_scores] + seen_by_score[unique_scores], group_lengths\n            )\n            ordered_ranks = rank_bases + offsets_in_score_group\n            ordered_band = np.searchsorted(\n                cut_counts, ordered_ranks, side="right"\n            ).astype(np.uint8)\n\n            band = np.empty(len(ordered_band), dtype=np.uint8)\n            band[order] = ordered_band\n            seen_by_score[unique_scores] += group_lengths\n            realized += np.bincount(ordered_band, minlength=len(NEGATIVE_BAND_NAMES))\n\n            keys = data["region_key"][neg_rows].astype(np.uint64, copy=False)\n            base_logits = logits[neg_rows, neg_offsets].astype(np.float32, copy=False)\n\n            for b in range(len(NEGATIVE_BAND_NAMES)):\n                where = np.flatnonzero(band == b)\n                if len(where) == 0:\n                    continue\n                # Keep at most K candidates from this shard before merging.\n                hashes = coordinate_hash(\n                    keys[where],\n                    neg_offsets[where],\n                    int(RNG_SEED + np.uint64(b) * np.uint64(10007)),\n                )\n                if len(where) > NEGATIVE_SAMPLE_PER_BAND:\n                    local = np.argpartition(hashes, NEGATIVE_SAMPLE_PER_BAND - 1)[\n                        :NEGATIVE_SAMPLE_PER_BAND\n                    ]\n                    where = where[local]\n                    hashes = hashes[local]\n\n                contigs, strands, positions = _decode_coordinates(\n                    keys[where], neg_offsets[where], all_contigs\n                )\n                chunk = pd.DataFrame(\n                    {\n                        "kind": "negative",\n                        "group": NEGATIVE_BAND_NAMES[b],\n                        "contig": contigs,\n                        "strand": strands,\n                        "position": positions.astype(np.int64),\n                        "base_logit": base_logits[where].astype(np.float64),\n                        "_hash": hashes.astype(np.uint64),\n                    }\n                )\n                merged = pd.concat([reservoirs[b], chunk], ignore_index=True)\n                reservoirs[b] = _trim_smallest(merged, NEGATIVE_SAMPLE_PER_BAND)\n\n        if file_index == 1 or file_index % 500 == 0 or file_index == len(files):\n            print(f"negative rank sampling: {file_index:,}/{len(files):,}", flush=True)\n\n    if not np.array_equal(realized, population_counts):\n        raise AssertionError(\n            f"Band population mismatch: realized={realized.tolist()} expected={population_counts.tolist()}"\n        )\n\n    negative_sample = pd.concat(\n        [reservoirs[i] for i in range(len(NEGATIVE_BAND_NAMES))],\n        ignore_index=True,\n    )\n    counts = negative_sample.groupby("group").size()\n    for band_name, pop_count in zip(NEGATIVE_BAND_NAMES, population_counts):\n        expected = min(NEGATIVE_SAMPLE_PER_BAND, int(pop_count))\n        actual = int(counts.get(band_name, 0))\n        if actual != expected:\n            raise AssertionError((band_name, actual, expected))\n\n    # Build a natural positive control, stratified by EXP045-score quartile.\n    pos_keys = np.concatenate(positive_keys_parts).astype(np.uint64, copy=False)\n    pos_offsets = np.concatenate(positive_offsets_parts).astype(np.uint8, copy=False)\n    pos_logits = np.concatenate(positive_logits_parts).astype(np.float32, copy=False)\n    pos_scores = sigmoid_numpy(pos_logits)\n    quartile_edges = np.quantile(pos_scores, [0.25, 0.50, 0.75])\n    pos_quartile = np.searchsorted(quartile_edges, pos_scores, side="right")\n    positive_frames = []\n\n    for q in range(4):\n        where = np.flatnonzero(pos_quartile == q)\n        hashes = coordinate_hash(\n            pos_keys[where],\n            pos_offsets[where],\n            int(RNG_SEED + np.uint64(1_000_000 + q * 10007)),\n        )\n        k = min(POSITIVE_SAMPLE_PER_QUARTILE, len(where))\n        if len(where) > k:\n            selected_local = np.argpartition(hashes, k - 1)[:k]\n            where = where[selected_local]\n            hashes = hashes[selected_local]\n        contigs, strands, positions = _decode_coordinates(\n            pos_keys[where], pos_offsets[where], all_contigs\n        )\n        positive_frames.append(\n            pd.DataFrame(\n                {\n                    "kind": "positive",\n                    "group": f"Q{q+1}",\n                    "contig": contigs,\n                    "strand": strands,\n                    "position": positions.astype(np.int64),\n                    "base_logit": pos_logits[where].astype(np.float64),\n                    "_hash": hashes.astype(np.uint64),\n                }\n            )\n        )\n\n    positive_sample = pd.concat(positive_frames, ignore_index=True)\n    sample = pd.concat([negative_sample, positive_sample], ignore_index=True)\n    sample["score_EXP045"] = sigmoid_numpy(sample["base_logit"].to_numpy(np.float64))\n    sample["target_start"] = sample["position"].to_numpy(np.int64) // 1024 * 1024\n    sample = sample.drop(columns=["_hash"]).sort_values(\n        ["kind", "group", "contig", "strand", "position"], kind="stable"\n    ).reset_index(drop=True)\n    sample.insert(0, "sample_id", np.arange(len(sample), dtype=np.int64))\n\n    if sample.duplicated(["kind", "contig", "strand", "position"]).any():\n        dup = sample.loc[\n            sample.duplicated(["kind", "contig", "strand", "position"], keep=False)\n        ].head()\n        raise AssertionError(f"Duplicate sampled coordinates:\\n{dup}")\n\n    meta = {\n        "complete": True,\n        "source_checkpoint": str(BASE_CHECKPOINT),\n        "source_checkpoint_sha256": sha256_file(BASE_CHECKPOINT),\n        "source_shards": str(CV_SHARDS),\n        "source_shard_count": len(files),\n        "score_definition_for_negative_rank": "sigmoid(EXP045 logit), rounded to 5 decimals",\n        "negative_tie_break": "stable shard/path/row/position order, exactly as EXP047 risk cache",\n        "negative_band_edges_fraction": NEGATIVE_BAND_EDGES.tolist(),\n        "negative_band_names": NEGATIVE_BAND_NAMES,\n        "negative_population_counts": population_counts.tolist(),\n        "negative_population_fractions": population_fractions.tolist(),\n        "negative_sample_per_band": NEGATIVE_SAMPLE_PER_BAND,\n        "positive_population_n": int(len(pos_scores)),\n        "positive_quartile_edges_EXP045_score": quartile_edges.tolist(),\n        "positive_sample_per_quartile": POSITIVE_SAMPLE_PER_QUARTILE,\n        "sample_rows": int(len(sample)),\n        "unique_tiles": int(sample[["contig", "target_start"]].drop_duplicates().shape[0]),\n        "elapsed_seconds": time.perf_counter() - started,\n    }\n    atomic_csv(SAMPLE_MANIFEST, sample)\n    atomic_json(SAMPLE_META, meta)\n    print(sample.groupby(["kind", "group"]).size())\n    print(json.dumps(meta, indent=2))\n\n\ndef _soft_gpu_pause(active_seconds: float) -> None:\n    if GPU_DUTY_CYCLE >= 0.999999:\n        return\n    time.sleep(max(float(active_seconds), 0.0) * (1.0 / GPU_DUTY_CYCLE - 1.0))\n\n\ndef score_model(label: str) -> None:\n    import torch\n    import torch_directml\n\n    if label == "EXP047B":\n        checkpoint_path = EXP047_CHECKPOINT\n        output_path = SCORES_047\n    elif label == "EXP048":\n        checkpoint_path = EXP048_CHECKPOINT\n        output_path = SCORES_048\n    else:\n        raise ValueError(label)\n\n    if output_path.exists():\n        existing = pd.read_csv(output_path)\n        sample = pd.read_csv(SAMPLE_MANIFEST)\n        if len(existing) == len(sample) and np.array_equal(\n            existing["sample_id"].to_numpy(np.int64),\n            sample["sample_id"].to_numpy(np.int64),\n        ):\n            print(f"{label} sample scores already complete: {output_path}")\n            return\n\n    sample = pd.read_csv(SAMPLE_MANIFEST)\n    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)\n    split = load_split(ROOT)\n    sequences, _ = load_baseline_sequences(split)\n    config = ProfileWindowConfig(**checkpoint["profile_config"])\n    generator = ProfileWindowGenerator(\n        prepared_split=split, sequences=sequences, part="train", config=config, seed=42\n    )\n    tile_map = {\n        (str(row.contig), int(row.target_start)): index\n        for index, row in generator.tile_index.iterrows()\n    }\n\n    grouped = sample.groupby(["contig", "target_start"], sort=True).groups\n    tile_groups = list(grouped.items())\n    missing = [key for key, _ in tile_groups if key not in tile_map]\n    if missing:\n        raise KeyError(f"Sample coordinates missing from train generator: {missing[:5]}")\n\n    device = torch_directml.device()\n    model = create_cnn_presence_intensity(channels=512)\n    model.load_state_dict(checkpoint["model_state_dict"])\n    model = model.to(device)\n    model.eval()\n\n    scores = np.full(len(sample), np.nan, dtype=np.float32)\n    batch_size = 4\n    started = time.perf_counter()\n\n    with torch.no_grad():\n        for start in range(0, len(tile_groups), batch_size):\n            chunk = tile_groups[start : start + batch_size]\n            indices = [tile_map[key] for key, _ in chunk]\n            selected = generator.tile_index.iloc[indices]\n            batch = generator.make_batch(selected)\n\n            phase_started = time.perf_counter()\n            with warnings.catch_warnings(record=True) as caught:\n                warnings.simplefilter("always")\n                plus, minus_reverse, _, _ = forward_both_strands_multitask(\n                    model,\n                    batch.x_both_strands.to(device),\n                    batch_size=len(chunk),\n                    context_flank=config.context_flank,\n                    target_length=config.target_length,\n                )\n                plus_np = sigmoid_numpy(plus[:, 0].detach().cpu().numpy())\n                minus_np = sigmoid_numpy(\n                    minus_reverse[:, 0].detach().cpu().numpy()[:, ::-1]\n                )\n            fallbacks = [\n                str(x.message)\n                for x in caught\n                if "fall back" in str(x.message).lower()\n            ]\n            if fallbacks:\n                raise RuntimeError("DirectML CPU fallback: " + " | ".join(fallbacks))\n\n            _soft_gpu_pause(time.perf_counter() - phase_started)\n\n            for local_batch, ((contig, target_start), row_ids) in enumerate(chunk):\n                row_ids = np.asarray(list(row_ids), dtype=np.int64)\n                locals_ = (\n                    sample.loc[row_ids, "position"].to_numpy(np.int64)\n                    - int(target_start)\n                )\n                strands = sample.loc[row_ids, "strand"].astype(str).to_numpy()\n                values = np.where(\n                    strands == "+",\n                    plus_np[local_batch, locals_],\n                    minus_np[local_batch, locals_],\n                )\n                scores[row_ids] = values.astype(np.float32)\n\n            completed = min(start + len(chunk), len(tile_groups))\n            if start == 0 or completed % 250 < len(chunk) or completed == len(tile_groups):\n                print(\n                    f"[{label}] tiles {completed:,}/{len(tile_groups):,} "\n                    f"elapsed={(time.perf_counter()-started)/60:.1f} min",\n                    flush=True,\n                )\n            del batch, plus, minus_reverse, plus_np, minus_np\n\n    if not np.all(np.isfinite(scores)):\n        bad = np.flatnonzero(~np.isfinite(scores))\n        raise AssertionError(f"Non-finite/missing scores: {bad[:20].tolist()}")\n\n    out = sample[["sample_id", "kind", "group", "contig", "strand", "position"]].copy()\n    out[f"score_{label}"] = scores\n    atomic_csv(output_path, out)\n    print(f"Saved: {output_path}")\n\n\ndef _safe_bce_negative(p: np.ndarray) -> np.ndarray:\n    p = np.clip(np.asarray(p, dtype=np.float64), 1e-12, 1.0 - 1e-12)\n    return -np.log1p(-p)\n\n\ndef _auc_pair(pos: np.ndarray, neg: np.ndarray) -> float:\n    # Mann-Whitney AUC with average ranks for ties, implemented without sklearn.\n    pos = np.asarray(pos, dtype=np.float64)\n    neg = np.asarray(neg, dtype=np.float64)\n    values = np.concatenate([pos, neg])\n    labels = np.concatenate([np.ones(len(pos), dtype=np.uint8), np.zeros(len(neg), dtype=np.uint8)])\n    order = np.argsort(values, kind="mergesort")\n    sorted_values = values[order]\n    ranks = np.empty(len(values), dtype=np.float64)\n    start = 0\n    while start < len(values):\n        end = start + 1\n        while end < len(values) and sorted_values[end] == sorted_values[start]:\n            end += 1\n        avg_rank = 0.5 * ((start + 1) + end)\n        ranks[order[start:end]] = avg_rank\n        start = end\n    rank_sum_pos = ranks[labels == 1].sum()\n    n_pos, n_neg = len(pos), len(neg)\n    return float((rank_sum_pos - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg))\n\n\ndef _bootstrap_mean_ci(values: np.ndarray, seed: int, n_boot: int = 1_000):\n    values = np.asarray(values, dtype=np.float64)\n    rng = np.random.default_rng(seed)\n    means = np.empty(n_boot, dtype=np.float64)\n    for i in range(n_boot):\n        means[i] = rng.choice(values, size=len(values), replace=True).mean()\n    return float(np.quantile(means, 0.025)), float(np.quantile(means, 0.975))\n\n\ndef analyze() -> None:\n    required = [SAMPLE_MANIFEST, SAMPLE_META, SCORES_047, SCORES_048]\n    missing = [p for p in required if not p.is_file()]\n    if missing:\n        raise FileNotFoundError(missing)\n\n    sample = pd.read_csv(SAMPLE_MANIFEST)\n    meta = json.loads(SAMPLE_META.read_text(encoding="utf-8"))\n    s47 = pd.read_csv(SCORES_047)[["sample_id", "score_EXP047B"]]\n    s48 = pd.read_csv(SCORES_048)[["sample_id", "score_EXP048"]]\n    frame = sample.merge(s47, on="sample_id", validate="one_to_one").merge(\n        s48, on="sample_id", validate="one_to_one"\n    )\n    atomic_csv(RUN_DIR / "train_tail_scored_sample.csv", frame)\n\n    score_cols = ["score_EXP045", "score_EXP047B", "score_EXP048"]\n    negative = frame.loc[frame["kind"] == "negative"].copy()\n    positive = frame.loc[frame["kind"] == "positive"].copy()\n\n    population = pd.DataFrame(\n        {\n            "group": meta["negative_band_names"],\n            "population_n": meta["negative_population_counts"],\n            "population_fraction": meta["negative_population_fractions"],\n        }\n    )\n\n    neg_rows = []\n    for group, part in negative.groupby("group", sort=False):\n        row = {"group": group, "sample_n": len(part)}\n        for col in score_cols:\n            x = part[col].to_numpy(np.float64)\n            suffix = col.removeprefix("score_")\n            row[f"mean_{suffix}"] = float(x.mean())\n            row[f"median_{suffix}"] = float(np.median(x))\n            row[f"p90_{suffix}"] = float(np.quantile(x, 0.90))\n            row[f"frac_gt_0p9_{suffix}"] = float((x > 0.9).mean())\n            row[f"frac_gt_0p5_{suffix}"] = float((x > 0.5).mean())\n            row[f"mean_neg_BCE_{suffix}"] = float(_safe_bce_negative(x).mean())\n        d47 = part["score_EXP047B"].to_numpy(np.float64) - part["score_EXP045"].to_numpy(np.float64)\n        d48 = part["score_EXP048"].to_numpy(np.float64) - part["score_EXP047B"].to_numpy(np.float64)\n        row["mean_delta_047_vs_045"] = float(d47.mean())\n        row["mean_delta_048_vs_047"] = float(d48.mean())\n        row["frac_lower_047_vs_045"] = float((d47 < 0).mean())\n        row["frac_lower_048_vs_047"] = float((d48 < 0).mean())\n        lo, hi = _bootstrap_mean_ci(d48, seed=4800 + NEGATIVE_BAND_NAMES.index(group))\n        row["delta_048_vs_047_ci95_low"] = lo\n        row["delta_048_vs_047_ci95_high"] = hi\n        neg_rows.append(row)\n\n    neg_summary = pd.DataFrame(neg_rows).merge(population, on="group", validate="one_to_one")\n    neg_summary["population_weighted_BCE_EXP045"] = (\n        neg_summary["population_fraction"] * neg_summary["mean_neg_BCE_EXP045"]\n    )\n    neg_summary["population_weighted_BCE_EXP047B"] = (\n        neg_summary["population_fraction"] * neg_summary["mean_neg_BCE_EXP047B"]\n    )\n    neg_summary["population_weighted_BCE_EXP048"] = (\n        neg_summary["population_fraction"] * neg_summary["mean_neg_BCE_EXP048"]\n    )\n    band_order = {name: i for i, name in enumerate(NEGATIVE_BAND_NAMES)}\n    neg_summary["_order"] = neg_summary["group"].map(band_order)\n    neg_summary = neg_summary.sort_values("_order").drop(columns="_order")\n    atomic_csv(RUN_DIR / "negative_band_summary.csv", neg_summary)\n\n    pos_rows = []\n    for group, part in positive.groupby("group", sort=False):\n        row = {"group": group, "sample_n": len(part)}\n        for col in score_cols:\n            x = part[col].to_numpy(np.float64)\n            suffix = col.removeprefix("score_")\n            row[f"mean_{suffix}"] = float(x.mean())\n            row[f"median_{suffix}"] = float(np.median(x))\n            row[f"p10_{suffix}"] = float(np.quantile(x, 0.10))\n        d47 = part["score_EXP047B"].to_numpy(np.float64) - part["score_EXP045"].to_numpy(np.float64)\n        d48 = part["score_EXP048"].to_numpy(np.float64) - part["score_EXP047B"].to_numpy(np.float64)\n        row["mean_delta_047_vs_045"] = float(d47.mean())\n        row["mean_delta_048_vs_047"] = float(d48.mean())\n        row["frac_higher_047_vs_045"] = float((d47 > 0).mean())\n        row["frac_higher_048_vs_047"] = float((d48 > 0).mean())\n        lo, hi = _bootstrap_mean_ci(d48, seed=4900 + int(group[1:]))\n        row["delta_048_vs_047_ci95_low"] = lo\n        row["delta_048_vs_047_ci95_high"] = hi\n        pos_rows.append(row)\n    pos_summary = pd.DataFrame(pos_rows).sort_values("group")\n    atomic_csv(RUN_DIR / "positive_quartile_summary.csv", pos_summary)\n\n    # Pairwise ranking separation: positives should outrank each negative band.\n    separation_rows = []\n    positive_sets = {\n        "all_positive_quartiles": positive,\n        "weak_positive_Q1": positive.loc[positive["group"] == "Q1"],\n    }\n    for pos_name, pos_part in positive_sets.items():\n        for neg_group in NEGATIVE_BAND_NAMES:\n            neg_part = negative.loc[negative["group"] == neg_group]\n            pop_frac = float(\n                population.loc[population["group"] == neg_group, "population_fraction"].iloc[0]\n            )\n            row = {\n                "positive_set": pos_name,\n                "negative_group": neg_group,\n                "negative_population_fraction": pop_frac,\n            }\n            for col in score_cols:\n                suffix = col.removeprefix("score_")\n                auc = _auc_pair(\n                    pos_part[col].to_numpy(np.float64),\n                    neg_part[col].to_numpy(np.float64),\n                )\n                row[f"AUC_{suffix}"] = auc\n                row[f"inversion_rate_{suffix}"] = 1.0 - auc\n                row[f"weighted_inversion_{suffix}"] = pop_frac * (1.0 - auc)\n            row["delta_AUC_047_vs_045"] = row["AUC_EXP047B"] - row["AUC_EXP045"]\n            row["delta_AUC_048_vs_047"] = row["AUC_EXP048"] - row["AUC_EXP047B"]\n            separation_rows.append(row)\n    separation = pd.DataFrame(separation_rows)\n    separation["_order"] = separation["negative_group"].map(band_order)\n    separation = separation.sort_values(["positive_set", "_order"]).drop(columns="_order")\n    atomic_csv(RUN_DIR / "negative_vs_positive_separation.csv", separation)\n\n    # Aggregate the exact user hypothesis zone 0.2-1%.\n    zone_parts = negative.loc[negative["group"].isin(["0.2-0.5%", "0.5-1%"])].copy()\n    zone_weights = {\n        "0.2-0.5%": 0.003,\n        "0.5-1%": 0.005,\n    }\n    # Normalize sample weights so the sampled 0.2-1% zone represents its natural 0.8% composition.\n    zone_parts["_natural_weight"] = zone_parts["group"].map(\n        {\n            g: w / max(1, int((zone_parts["group"] == g).sum()))\n            for g, w in zone_weights.items()\n        }\n    )\n    # Weighted score means are enough here; pairwise AUC remains reported separately by sub-band.\n    zone_summary = {\n        "zone": "0.2-1%",\n        "natural_fraction": 0.008,\n        "sample_n": int(len(zone_parts)),\n    }\n    w = zone_parts["_natural_weight"].to_numpy(np.float64)\n    w = w / w.sum()\n    for col in score_cols:\n        zone_summary[f"weighted_mean_{col.removeprefix(\'score_\')}"] = float(\n            np.sum(w * zone_parts[col].to_numpy(np.float64))\n        )\n    zone_summary["weighted_mean_delta_048_vs_047"] = (\n        zone_summary["weighted_mean_EXP048"] - zone_summary["weighted_mean_EXP047B"]\n    )\n    atomic_csv(RUN_DIR / "user_hypothesis_zone_0p2_1_summary.csv", pd.DataFrame([zone_summary]))\n\n    # Existing validation fixed cohorts: no new validation inference.\n    validation_summary = []\n    if EXP047_COHORT.is_file() and EXP048_COHORT.is_file():\n        v47 = pd.read_csv(EXP047_COHORT).rename(columns={"score_B": "score_EXP047B"})\n        v48 = pd.read_csv(EXP048_COHORT).rename(columns={"score_B": "score_EXP048"})\n        keys = ["contig", "strand", "position", "baseline_score", "cohort"]\n        merged = v47[keys + ["score_EXP047B"]].merge(\n            v48[keys + ["score_EXP048"]], on=keys, validate="one_to_one"\n        )\n        merged["delta_048_vs_047"] = merged["score_EXP048"] - merged["score_EXP047B"]\n        atomic_csv(RUN_DIR / "validation_fixed_cohort_EXP047B_vs_EXP048.csv", merged)\n        validation_summary = (\n            merged.groupby("cohort", as_index=False)\n            .agg(\n                n=("delta_048_vs_047", "size"),\n                mean_EXP047B=("score_EXP047B", "mean"),\n                mean_EXP048=("score_EXP048", "mean"),\n                mean_delta=("delta_048_vs_047", "mean"),\n                frac_higher_048=("delta_048_vs_047", lambda x: float((x > 0).mean())),\n                frac_lower_048=("delta_048_vs_047", lambda x: float((x < 0).mean())),\n            )\n        )\n        atomic_csv(RUN_DIR / "validation_fixed_cohort_summary.csv", validation_summary)\n\n    # Machine-readable diagnostic facts, not a decision to train another model.\n    weak_sep = separation.loc[separation["positive_set"] == "weak_positive_Q1"].copy()\n    strongest_047 = weak_sep.sort_values("delta_AUC_047_vs_045", ascending=False).iloc[0]\n    worst_048 = weak_sep.sort_values("delta_AUC_048_vs_047", ascending=True).iloc[0]\n    facts = {\n        "complete": True,\n        "purpose": (\n            "Post-hoc causal diagnostic of where EXP047 helped and EXP048 hurt; "\n            "not a new loss selection by itself."\n        ),\n        "train_negative_bands": NEGATIVE_BAND_NAMES,\n        "sample_rows": int(len(frame)),\n        "unique_tiles": int(frame[["contig", "target_start"]].drop_duplicates().shape[0]),\n        "strongest_EXP047_AUC_gain_vs_EXP045_for_weak_positives": {\n            "band": str(strongest_047["negative_group"]),\n            "delta_AUC": float(strongest_047["delta_AUC_047_vs_045"]),\n        },\n        "largest_EXP048_AUC_damage_vs_EXP047_for_weak_positives": {\n            "band": str(worst_048["negative_group"]),\n            "delta_AUC": float(worst_048["delta_AUC_048_vs_047"]),\n        },\n        "user_hypothesis_zone_0p2_1": zone_summary,\n        "validation_fixed_cohorts_reused_without_new_validation_inference": bool(\n            EXP047_COHORT.is_file() and EXP048_COHORT.is_file()\n        ),\n        "files": {\n            "negative_band_summary": str(RUN_DIR / "negative_band_summary.csv"),\n            "positive_quartile_summary": str(RUN_DIR / "positive_quartile_summary.csv"),\n            "separation": str(RUN_DIR / "negative_vs_positive_separation.csv"),\n            "zone_0p2_1": str(RUN_DIR / "user_hypothesis_zone_0p2_1_summary.csv"),\n            "validation_fixed_cohort_summary": str(RUN_DIR / "validation_fixed_cohort_summary.csv"),\n        },\n    }\n    atomic_json(RUN_DIR / "diagnostic_summary.json", facts)\n\n    print("\\n=== NEGATIVE BANDS ===")\n    print(\n        neg_summary[\n            [\n                "group",\n                "population_n",\n                "population_fraction",\n                "mean_EXP045",\n                "mean_EXP047B",\n                "mean_EXP048",\n                "mean_delta_047_vs_045",\n                "mean_delta_048_vs_047",\n                "frac_gt_0p9_EXP047B",\n                "frac_gt_0p9_EXP048",\n            ]\n        ].to_string(index=False)\n    )\n    print("\\n=== POSITIVE QUARTILES ===")\n    print(pos_summary.to_string(index=False))\n    print("\\n=== WEAK POSITIVE Q1 SEPARATION ===")\n    print(\n        weak_sep[\n            [\n                "negative_group",\n                "AUC_EXP045",\n                "AUC_EXP047B",\n                "AUC_EXP048",\n                "delta_AUC_047_vs_045",\n                "delta_AUC_048_vs_047",\n            ]\n        ].to_string(index=False)\n    )\n    print("\\n=== 0.2-1% USER HYPOTHESIS ZONE ===")\n    print(pd.DataFrame([zone_summary]).to_string(index=False))\n    if isinstance(validation_summary, pd.DataFrame) and not validation_summary.empty:\n        print("\\n=== EXISTING VALIDATION FIXED COHORTS ===")\n        print(validation_summary.to_string(index=False))\n    print(f"\\nSaved diagnostics under: {RUN_DIR}")\n\n\ndef parse_args():\n    p = argparse.ArgumentParser()\n    sub = p.add_subparsers(dest="command", required=True)\n    sub.add_parser("build-sample")\n    score = sub.add_parser("score-model")\n    score.add_argument("--model", choices=["EXP047B", "EXP048"], required=True)\n    sub.add_parser("analyze")\n    return p.parse_args()\n\n\ndef main():\n    args = parse_args()\n    if args.command == "build-sample":\n        build_sample()\n    elif args.command == "score-model":\n        score_model(args.model)\n    elif args.command == "analyze":\n        analyze()\n    else:\n        raise AssertionError(args.command)\n\n\nif __name__ == "__main__":\n    main()\n'
EXPECTED_SCRIPT_SHA = "4998f0c24ee9ee0df731f36e850f38a171fb7deb7fcf491696790eedae7968fa"
actual_source_sha = hashlib.sha256(SCRIPT_SOURCE.encode("utf-8")).hexdigest()
assert actual_source_sha == EXPECTED_SCRIPT_SHA
compile(SCRIPT_SOURCE, str(TARGET_SCRIPT), "exec")

if TARGET_SCRIPT.exists():
    existing = hashlib.sha256(TARGET_SCRIPT.read_bytes()).hexdigest()
    assert existing == EXPECTED_SCRIPT_SHA, (
        f"Existing diagnostic script differs: {existing} != {EXPECTED_SCRIPT_SHA}"
    )
else:
    TARGET_SCRIPT.write_text(SCRIPT_SOURCE, encoding="utf-8")

RUN_DIR = PROJECT_ROOT / "artifacts/experiments/CNN_EXP048_TAIL_DIAGNOSTIC/run_001"
env = os.environ.copy()
env["EPIC_GPU_DUTY_CYCLE"] = "0.90"

def run_stage(*args):
    cmd = [str(EXPECTED_PYTHON), "-u", str(TARGET_SCRIPT), *map(str, args)]
    print("RUN:", subprocess.list2cmdline(cmd), flush=True)
    p = subprocess.run(cmd, cwd=PROJECT_ROOT, env=env)
    if p.returncode != 0:
        raise RuntimeError((p.returncode, cmd))

print("Diagnostic script SHA256:", EXPECTED_SCRIPT_SHA)
print("Artifacts:", RUN_DIR)


AGENTS.md: D:\Projects\EPIC\AGENTS.md
# Project agent instructions

- Run all Python commands, notebooks, analyses, and tests for this workspace in the Conda environment `epic-eda`.
- On this Windows machine, prefer the explicit interpreter `C:\ML_Progs\Runtimes\Miniforge3\envs\epic-eda\python.exe` when invoking Python non-interactively.
- Do not use the repository `.venv` or the system Python for project work unless the user explicitly requests it.

## Decision-making and scientific judgment

- Do not automatically agree with the user's proposed interpretation, experiment, architecture, or next step.
- Treat the user's preferred direction as a hypothesis to evaluate, not as the conclusion to defend.
- Independently compare the available alternatives using experimental evidence, expected information gain, methodological validity, cost, risk, and relevance to the primary objective.
- Recommend the path that is most logically supported by the evidence, even when it conflicts with the use

## 1. Build the fixed fine-tail sample

CPU-only. It scans the already saved corrected EXP045 train shards twice:

1. exact global 5-decimal score histogram + positive population;
2. exact stable rank bands + deterministic coordinate sample.

No model forward is performed here.


In [2]:
run_stage("build-sample")

meta = json.loads((RUN_DIR/"sample/train_tail_sample_meta.json").read_text(encoding="utf-8"))
display(pd.DataFrame({
    "band": meta["negative_band_names"],
    "population_n": meta["negative_population_counts"],
    "population_fraction": meta["negative_population_fractions"],
}))
print("sample_rows:", meta["sample_rows"])
print("unique_tiles:", meta["unique_tiles"])


RUN: C:\ML_Progs\Runtimes\Miniforge3\envs\epic-eda\python.exe -u D:\Projects\EPIC\EPIC\scripts\run_exp048_tail_diagnostic.py build-sample


,band,population_n,population_fraction
0,0-0.1%,227894,0.001
1,0.1-0.2%,227893,0.001
2,0.2-0.5%,683682,0.003
3,0.5-1%,1139468,0.005
4,1-5%,9115749,0.040
5,5-15%,22789372,0.100
6,15-100%,193709659,0.850


sample_rows: 11000
unique_tiles: 9555


## 2. Score the exact same train coordinates with EXP047-B

Fresh subprocess + DirectML. Soft duty-cycle target = 0.90.


In [3]:
run_stage("score-model", "--model", "EXP047B")


RUN: C:\ML_Progs\Runtimes\Miniforge3\envs\epic-eda\python.exe -u D:\Projects\EPIC\EPIC\scripts\run_exp048_tail_diagnostic.py score-model --model EXP047B


## 3. Score the exact same train coordinates with EXP048


In [4]:
run_stage("score-model", "--model", "EXP048")


RUN: C:\ML_Progs\Runtimes\Miniforge3\envs\epic-eda\python.exe -u D:\Projects\EPIC\EPIC\scripts\run_exp048_tail_diagnostic.py score-model --model EXP048


## 4. Analyze fine bands, weak positives and ranking inversions

This stage is CPU-only and writes all diagnostic tables.


In [5]:
run_stage("analyze")


RUN: C:\ML_Progs\Runtimes\Miniforge3\envs\epic-eda\python.exe -u D:\Projects\EPIC\EPIC\scripts\run_exp048_tail_diagnostic.py analyze


## 5. Read the result tables

Pay particular attention to:

- `mean_delta_048_vs_047` — where EXP048 pushed negatives relative to EXP047;
- `delta_AUC_048_vs_047` against `weak_positive_Q1` — where ultra-tail weighting damaged/helped ordering;
- `weighted_inversion_*` — natural-population-weighted ranking inversion proxy;
- `user_hypothesis_zone_0p2_1_summary.csv` — direct summary of the proposed 0.2–1% zone.


In [6]:
negative = pd.read_csv(RUN_DIR/"negative_band_summary.csv")
positive = pd.read_csv(RUN_DIR/"positive_quartile_summary.csv")
separation = pd.read_csv(RUN_DIR/"negative_vs_positive_separation.csv")
zone = pd.read_csv(RUN_DIR/"user_hypothesis_zone_0p2_1_summary.csv")
summary = json.loads((RUN_DIR/"diagnostic_summary.json").read_text(encoding="utf-8"))

print("=== NEGATIVE BANDS ===")
display(negative[[
    "group","population_n","population_fraction",
    "mean_EXP045","mean_EXP047B","mean_EXP048",
    "mean_delta_047_vs_045","mean_delta_048_vs_047",
    "frac_gt_0p9_EXP047B","frac_gt_0p9_EXP048",
    "population_weighted_BCE_EXP047B","population_weighted_BCE_EXP048",
]])

print("=== POSITIVE QUARTILES ===")
display(positive)

print("=== WEAK POSITIVE Q1 VS NEGATIVE BANDS ===")
display(separation.loc[separation["positive_set"]=="weak_positive_Q1", [
    "negative_group","negative_population_fraction",
    "AUC_EXP045","AUC_EXP047B","AUC_EXP048",
    "delta_AUC_047_vs_045","delta_AUC_048_vs_047",
    "weighted_inversion_EXP047B","weighted_inversion_EXP048",
]])

print("=== USER HYPOTHESIS: 0.2–1% ===")
display(zone)

validation_summary_path = RUN_DIR/"validation_fixed_cohort_summary.csv"
if validation_summary_path.is_file():
    print("=== EXISTING VALIDATION FIXED COHORTS ===")
    display(pd.read_csv(validation_summary_path))

print("=== MACHINE-READABLE FACTS ===")
print(json.dumps(summary, ensure_ascii=False, indent=2))


=== NEGATIVE BANDS ===


,group,population_n,population_fraction,mean_EXP045,mean_EXP047B,mean_EXP048,mean_delta_047_vs_045,mean_delta_048_vs_047,frac_gt_0p9_EXP047B,frac_gt_0p9_EXP048,population_weighted_BCE_EXP047B,population_weighted_BCE_EXP048
0,0-0.1%,227894,0.001,0.997701,0.927563,0.861695,-0.070139,-0.065868,0.773,0.333,0.002904,0.002101
1,0.1-0.2%,227893,0.001,0.992630,0.858767,0.824286,-0.133863,-0.034481,0.285,0.117,0.002070,0.001824
2,0.2-0.5%,683682,0.003,0.982401,0.791648,0.784355,-0.190753,-0.007293,0.066,0.037,0.004945,0.004826
3,0.5-1%,1139468,0.005,0.961672,0.716612,0.733755,-0.245061,0.017143,0.006,0.003,0.006555,0.006888
4,1-5%,9115749,0.040,0.838143,0.532538,0.566706,-0.305605,0.034168,0.001,0.000,0.031925,0.035267
5,5-15%,22789372,0.100,0.496202,0.298256,0.322573,-0.197946,0.024316,0.000,0.000,0.036595,0.040557
6,15-100%,193709659,0.850,0.050436,0.048244,0.049602,-0.002192,0.001358,0.000,0.000,0.044048,0.045574


=== POSITIVE QUARTILES ===


,group,sample_n,mean_EXP045,median_EXP045,p10_EXP045,mean_EXP047B,median_EXP047B,p10_EXP047B,mean_EXP048,median_EXP048,p10_EXP048,mean_delta_047_vs_045,mean_delta_048_vs_047,frac_higher_047_vs_045,frac_higher_048_vs_047,delta_048_vs_047_ci95_low,delta_048_vs_047_ci95_high
0,Q1,1000,0.623605,0.706126,0.194963,0.478653,0.507194,0.196123,0.504871,0.546328,0.209490,-0.144952,0.026218,0.181,0.780,0.023827,0.028571
1,Q2,1000,0.937090,0.943256,0.896033,0.730034,0.739886,0.599859,0.740805,0.750741,0.625489,-0.207056,0.010771,0.003,0.641,0.007871,0.013660
2,Q3,1000,0.987583,0.988641,0.977372,0.869629,0.883523,0.773599,0.836839,0.845208,0.750018,-0.117954,-0.032790,0.000,0.220,-0.035359,-0.030208
3,Q4,1000,0.998866,0.999174,0.997244,0.969494,0.977961,0.932434,0.911790,0.921278,0.850874,-0.029372,-0.057705,0.000,0.007,-0.059505,-0.055891


=== WEAK POSITIVE Q1 VS NEGATIVE BANDS ===


...

=== USER HYPOTHESIS: 0.2–1% ===


,zone,natural_fraction,sample_n,weighted_mean_EXP045,weighted_mean_EXP047B,weighted_mean_EXP048,weighted_mean_delta_048_vs_047
0,0.2-1%,0.008,2000,0.969446,0.74475,0.75273,0.007979


=== EXISTING VALIDATION FIXED COHORTS ===


,cohort,n,mean_EXP047B,mean_EXP048,mean_delta,frac_higher_048,frac_lower_048
0,HARD_FP,20000,0.937258,0.889905,-0.047353,0.07710,0.92290
1,LOW_TP,14052,0.239948,0.259912,0.019964,0.67179,0.32821


=== MACHINE-READABLE FACTS ===
{
  "complete": true,
  "purpose": "Post-hoc causal diagnostic of where EXP047 helped and EXP048 hurt; not a new loss selection by itself.",
  "train_negative_bands": [
    "0-0.1%",
    "0.1-0.2%",
    "0.2-0.5%",
    "0.5-1%",
    "1-5%",
    "5-15%",
    "15-100%"
  ],
  "sample_rows": 11000,
  "unique_tiles": 9555,
  "strongest_EXP047_AUC_gain_vs_EXP045_for_weak_positives": {
    "band": "1-5%",
    "delta_AUC": 0.2420335
  },
  "largest_EXP048_AUC_damage_vs_EXP047_for_weak_positives": {
    "band": "5-15%",
    "delta_AUC": -0.0068590000000000595
  },
  "user_hypothesis_zone_0p2_1": {
    "zone": "0.2-1%",
    "natural_fraction": 0.008,
    "sample_n": 2000,
    "weighted_mean_EXP045": 0.9694457219419282,
    "weighted_mean_EXP047B": 0.74475033451875,
    "weighted_mean_EXP048": 0.7527298279300001,
    "weighted_mean_delta_048_vs_047": 0.00797949341125015
  },
  "validation_fixed_cohorts_reused_without_new_validation_inference": true,
  "files": {
  